# Subprocesses

This example notebook demonstrates how to create subprocesses that can be called from other processes.

In particular, this example covers:

- Creating processes/behavior trees with parameters
    - Creating a signature
    - Using parameters within the process
    - Returning values from the process
- Directly running a process with fixed parameters
- Calling a process with parameters from another process

<div class="alert alert-info">

**Important**

This notebook requires a running Flowstate solution to connect to. To start a solution:

1. Navigate to [flowstate.intrinsic.ai](https://flowstate.intrinsic.ai/) and sign in
   using your registered Flowstate account.

1. Do **one** of the following:
    - Create a new solution:
        1. Click "Create new solution" and choose "From an example".
        1. Select `pick_and_place:pick_and_place_module2`
        1. Click "Create".
    - Or open an existing solution that was created from the `pick_and_place:pick_and_place_module2` example:
        1. Hover over the solution in the list.
        1. Click "Open solution" or "Start solution".

1. Recommended: Keep the browser tab with the Flowstate solution editor open to watch the effect of notebook actions such as running a skill. You can simultaneously interact with the solution through the web UI and the notebook.

</div>

## Preparation

First, connect and define convenience shortcuts.

In [ ]:
from intrinsic.math.python import data_types
from intrinsic.solutions import behavior_tree as bt
from intrinsic.solutions import proto_building as pb
from intrinsic.solutions import deployments
from intrinsic.solutions import worlds

solution = deployments.connect_to_selected_solution()

executive = solution.executive
resources = solution.resources
skills = solution.skills
world = solution.world
proto_builder = solution.proto_builder

attach_object_to_robot = skills.ai.intrinsic.attach_object_to_robot
capture_images = skills.ai.intrinsic.capture_images
control_suction_gripper = skills.ai.intrinsic.control_suction_gripper
detach_object = skills.ai.intrinsic.detach_object
enable_gripper = skills.ai.intrinsic.enable_gripper
estimate_and_update_pose = skills.ai.intrinsic.estimate_and_update_pose
move_robot = skills.ai.intrinsic.move_robot
plan_grasp = skills.ai.intrinsic.plan_grasp

Create an initialization tree.

In [ ]:
init_gripper = enable_gripper(gripper=resources.picobot_gripper)

move_home = move_robot(
    motion_segments=[
        move_robot.intrinsic_proto.skills.MotionSegment(
            joint_position=world.robot.joint_configurations.home,
            motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
        )
    ],
    arm_part=world.robot,
)

initialize = bt.SubTree(
    name="Initialize",
    behavior_tree=bt.Sequence([
        bt.Task(action=init_gripper, name="Enable gripper"),
        bt.Task(action=move_home, name="Move home"),
    ]),
)


Create generic task nodes that we will use below.

In [ ]:
collisions_disabled = move_robot.intrinsic_proto.world.CollisionSettings(
    disable_collision_checking=True
)

# Move up 3cm from the current position without collision checking.
move_up_unsafe = bt.Task(
    name="Move up a bit",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                relative_cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.RelativePoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    relative_pose=data_types.Pose3(translation=[0, 0, -0.03]),
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.LINEAR,
                collision_settings=collisions_disabled,
            )
        ],
        arm_part=world.robot,
    )
)

# Turn on the suction gripper which will "attach" the building block in the
# simulation/real world.
gripper_grasp = bt.Task(
    name="Grasp gripper",
    action=control_suction_gripper(
        grasp=control_suction_gripper.intrinsic_proto.eoat.GraspRequest()
    )
)

# Turn off the suction gripper which will "drop" the building block in the
# simulation/real world.
gripper_release = bt.Task(
    name="Release gripper",
    action=control_suction_gripper(
        release=control_suction_gripper.intrinsic_proto.eoat.ReleaseRequest()
    )
)

## Create a signature

Processes can have inputs and outputs. This works very similarly to skills in that the output and input of each process is defined by a *parameter* and *return value* Protobuf message. These messages can be created with the help of `solution.proto_builder.create_signature()` which creates a `Signature` object that can be passed to `BehaviorTree.set_signature()`.

Here is an example signature for a process which has three input parameters (`my_int`, `my_string`, `my_joint_config`) and one return value (`my_frame`).

In [ ]:
my_first_signature = proto_builder.create_signature(
    parameters=pb.MessageSpec(fields=[
        pb.FieldSpec(type="int64",
                     name="my_int", number=1,
                     description=("Description which gets, e.g., shown in the "
                                  "frontend and included in Python stubs")),
        pb.FieldSpec(type="string",
                     name="my_string", number=2),
        pb.FieldSpec(type="intrinsic_proto.icon.JointVec",
                     name="my_joint_config", number=3),
    ]),
    return_value=pb.MessageSpec(fields=[
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="my_frame", number=1),
    ]),
)

The solution's `proto_builder` internally creates an actual proto file from the given message specs and runs `protoc` on it. The `MessageSpec` and `FieldSpec` classes offer a structured way to define the two messages required for a signature. The parameters of the `FieldSpec` class directly map to elements of the [proto syntax](https://protobuf.dev/programming-guides/proto3/#simple). Just as in `.proto` files, you must not only specify a *name* for each field but also a field [*number*](https://protobuf.dev/programming-guides/proto3/#assigning) which should be unique and never be reused when you add or remove fields.

The field *type* can be either a [Protobuf scalar type](https://protobuf.dev/programming-guides/proto3/#scalar) (`string`, `int64`, `bool`, `double`, ...) or one of the "well-known" types supported by the proto builder. These include several [Protobuf well-known types](https://protobuf.dev/reference/protobuf/google.protobuf/) as well as the most important protos from the Intrinsic SDK. You can get an up-to-date list of supported types with `proto_builder.get_well_known_types()`.

In [ ]:
proto_builder.get_well_known_types()

## Create a subprocess to plan a grasp

The first subprocess in this notebook estimates the pose of the building block object, creates a suitable pre-grasp and grasp frame and then returns two references to these frames called `pre_grasp_frame` and `grasp_frame`. The process takes as parameters a joint position `view_joint_config` from which the pose estimation should be performed and a reference called `block` to the building block object. These parameters are available via `estimate_and_plan_tree.params.<param>` and can be used in the same way that one would use a skill return value.

In [ ]:
estimate_and_plan_signature = proto_builder.create_signature(
    parameters=pb.MessageSpec(fields=[
        pb.FieldSpec(type="intrinsic_proto.icon.JointVec",
                     name="view_joint_config", number=1,
                     description="Joint position from which the block is visible"),
        pb.FieldSpec(type="intrinsic_proto.world.ObjectReference",
                     name="block", number=2,
                     description="Reference to block object"),
    ]),
    return_value=pb.MessageSpec(fields=[
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="pre_grasp_frame", number=1,
                     description="Resulting pre-grasp frame"),
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="grasp_frame", number=2,
                     description="Resulting grasp frame"),
    ]),
)

estimate_and_plan_tree = bt.BehaviorTree(name="Estimate block and plan grasp")
# subprocess=True makes the process accessible via solution.skills below and
# also makes the process show up in the list of available skills in the process
# editor in the frontend.
estimate_and_plan_tree.set_asset_metadata(
    id="com.myorg.estimate_and_plan", vendor="MyOrg", subprocess=True)
estimate_and_plan_tree.set_signature(estimate_and_plan_signature)

# Move such that the robot-mounted camera can see the object.
move_view_pose = bt.Task(
    name="Move to view pose",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                # Use `view_joint_config` from process parameters
                joint_position=estimate_and_plan_tree.params.view_joint_config,
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
            )
        ],
        arm_part=world.robot,
    ),
)

capture_image = bt.Task(
    name="Capture image",
    action=capture_images(camera=resources.wrist_camera),
)

estimate_and_update_block = bt.Task(
    name="Estimate and update block",
    action=estimate_and_update_pose(
        capture_data=capture_image.result.capture_data,
        pose_estimator=solution.pose_estimators.building_block_ml_estimator,
        # Use `block` from process parameters
        object=estimate_and_plan_tree.params.block,
    ),
)

# Plan a grasp. This skill creates a pre-grasp and a grasp frame in the world
# and returns references to them as part of its result.
plan_block_grasp = bt.Task(
    name="Plan block grasp",
    action=plan_grasp(
        candidate_objects=plan_grasp.intrinsic_proto.manipulation.skills.ObjectsTarget(
            # Use `block` from process parameters
            objects=[estimate_and_plan_tree.params.block]
        ),
        tool_frame=world.picobot_gripper.tool_frame,
        grasp_annotations=plan_grasp.intrinsic_proto.grasping.GraspAnnotations(
            box_shaped_grasp_annotations=plan_grasp.intrinsic_proto.grasping.BoxShapedGraspAnnotationParams(
                obj_dims_in_meters=plan_grasp.intrinsic_proto.Vector3(
                    x=0.075, y=0.025, z=0.015
                ),
                num_rotations=2,
            )
        ),
    ),
)

estimate_and_plan_tree.set_root(
    bt.Sequence([
        move_view_pose,
        capture_image,
        estimate_and_update_block,
        plan_block_grasp,
    ])
)

# Use Signature.create_return_value_expression() to define how the result value
# message should be created from the values available on the blackboard once the
# tree has run to the end.
estimate_and_plan_tree.return_value_expression = (
    estimate_and_plan_signature.create_return_value_expression(
        # Return the first pre-grasp frame from the 'plan_grasp' result as 'pre_grasp_frame'.
        pre_grasp_frame=plan_block_grasp.result.grasps[0].pre_grasp_frame,
        # Return the first grasp frame from the 'plan_grasp' result as 'grasp_frame'.
        grasp_frame=plan_block_grasp.result.grasps[0].grasp_frame,
    )
)

## Run a process with parameters

Now we can test our tree by running it. Because the tree has parameters, we need to specify concrete values for those parameters when passing the tree to `executive.run()`. For that purpose, the `Signature` class has a method `create_params_message()` for creating a parameter message from fixed values. The method raises an error if we try to use a blackboard parameter assignment, such as `create_params_message(view_joint_config=some_skill.result.computed_joint_position)`.

In [ ]:
executive.run(
    estimate_and_plan_tree,
    parameters=estimate_and_plan_signature.create_params_message(
        view_joint_config=world.robot.joint_configurations.view_pose_left,
        block=world.building_block0,
    ),
    start_from_world_state=worlds.EditWorldId.INITIAL,
)

# Print the result message with the pre-grasp and grasp frame
print("Operation result:")
print(executive.operation.result)

Note that we just ran the tree *directly* by passing it to `executive.run()`. This approach is suitable for testing. Once we are happy with the tree's functionality, we can run the process as an actual subprocess by saving it and calling it from other processes. This is shown below.

## Create a subprocess to pick and place

Our second subprocess works on top of the first one. It expects the pre-grasp and grasp frames returned by the first subprocess and uses them to pick up the block object. Then it will move to a position above the frame given by the `place_frame` parameter and release the object. This process does not have any return values.

In [ ]:
pick_and_place_signature = proto_builder.create_signature(
    parameters=pb.MessageSpec(fields=[
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="pre_grasp_frame", number=1,
                     description="Frame to use as pre-grasp pose"),
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="grasp_frame", number=2,
                     description="Frame to use as grasp pose"),
        pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                     name="place_frame", number=3,
                     description="Pose where to drop the block"),
        pb.FieldSpec(type="intrinsic_proto.world.ObjectReference",
                     name="block", number=4,
                     description="Reference to block object"),
    ]),
)

pick_and_place_tree = bt.BehaviorTree(name="Pick and place block")
pick_and_place_tree.set_asset_metadata(
    id="com.myorg.pick_and_place", vendor="MyOrg", subprocess=True)
pick_and_place_tree.set_signature(pick_and_place_signature)

move_pregrasp = bt.Task(
    name="Move to pre-grasp",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.PoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    # Use `pre_grasp_frame` from process parameters
                    target_frame=pick_and_place_tree.params.pre_grasp_frame,
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
            )
        ],
        arm_part=world.robot,
    ),
)

move_grasp_unsafe = bt.Task(
    name="Move to grasp",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.PoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    # Use `grasp_frame` from process parameters
                    target_frame=pick_and_place_tree.params.grasp_frame,
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.LINEAR,
                collision_settings=collisions_disabled,
            )
        ],
        arm_part=world.robot,
    ),
)

# Attach block to robot in the belief world. This does not activate the gripper.
attach_block = bt.Task(
    name="Attach block to robot",
    action=attach_object_to_robot(
        gripper_entity=world.picobot_gripper,
        # Use `block` from process parameters
        object_entity=pick_and_place_tree.params.block,
    ),
)

move_drop = bt.Task(
    name="Move to drop",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.PoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    # Use `place_frame` from process parameters
                    target_frame=pick_and_place_tree.params.place_frame,
                    target_frame_offset=data_types.Pose3(
                        rotation=data_types.Rotation3.from_euler_angles(
                            rpy_degrees=[180, 0, 0]
                        ),
                        translation=[0, 0, 0.05],
                    ),
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
            )
        ],
        arm_part=world.robot,
    ),
)

# Detach block from robot in the belief world. This does not deactivate the gripper.
detach_block = bt.Task(
    name="Detach block from robot",
    action=detach_object(
        gripper_entity=world.picobot_gripper,
        # Use `block` from process parameters
        object_entity=pick_and_place_tree.params.block,
    ),
)

pick_and_place_tree.set_root(
    bt.Sequence([
        move_pregrasp,
        move_grasp_unsafe,
        gripper_grasp,
        attach_block,
        move_up_unsafe,
        move_drop,
        gripper_release,
        detach_block,
        move_up_unsafe,
    ])
)

## Call a subprocess

To call our subprocesses from another process, we first need to save them to the solution. The subprocesses then become available through `solution.skills` once we re-connect to the solution or once we call `solution.update_skills()`.

In [ ]:
solution.processes.save(estimate_and_plan_tree)
solution.processes.save(pick_and_place_tree)
solution.update_skills()

# Should include 'com.myorg.estimate_and_plan' and 'com.myorg.pick_and_place'
list(solution.skills.get_skill_ids())

Now we can create and run a "main" process that calls our two subprocesses in sequence while passing the frames returned by the first subprocess to the second subprocess. This moves the block object from the left side to the right side.

In [ ]:
estimate_and_plan_1 = bt.Task(
    name="Estimate and plan left",
    action=skills.com.myorg.estimate_and_plan(
        # Initially the object is at the left side, so we need to move there for estimation
        view_joint_config=world.robot.joint_configurations.view_pose_left,
        block=world.building_block0,
    ),
)

pick_and_place_1 = bt.Task(
    name="Pick and place to right target",
    action=skills.com.myorg.pick_and_place(
        # Pass on frames returned by 'estimate_and_plan_1'
        pre_grasp_frame=estimate_and_plan_1.result.pre_grasp_frame,
        grasp_frame=estimate_and_plan_1.result.grasp_frame,
        # Move object to the other side (right)
        place_frame=world.target_right,
        block=world.building_block0,
    ),
)

executive.run(
    bt.BehaviorTree(
        name="Main Sequence 1",
        root=bt.Sequence([
            initialize,
            estimate_and_plan_1,
            pick_and_place_1,
        ]),
    ),
    start_from_world_state=worlds.EditWorldId.INITIAL,
)

Last but not least, we create and run a second "main" process. Like skills, we can re-use our subprocesses with other parameters. In this case, we use different parameters to move the block back from the right side to the left side.

In [ ]:
estimate_and_plan_2 = bt.Task(
    name="Estimate and plan right",
    action=skills.com.myorg.estimate_and_plan(
        view_joint_config=world.robot.joint_configurations.view_pose_right,
        block=world.building_block0,
    ),
)

pick_and_place_2 = bt.Task(
    name="Pick and place to left target",
    action=skills.com.myorg.pick_and_place(
        # Pass on frames returned by 'estimate_and_plan_2'
        pre_grasp_frame=estimate_and_plan_2.result.pre_grasp_frame,
        grasp_frame=estimate_and_plan_2.result.grasp_frame,
        # Move object to the other side (left)
        place_frame=world.target_left,
        block=world.building_block0,
    ),
)

# Run continuing from previous state (by omitting 'start_from_world_state')
executive.run(
    bt.BehaviorTree(
        name="Main Sequence 2",
        root=bt.Sequence([
            estimate_and_plan_2,
            pick_and_place_2,
        ]),
    ),
)

## Summary

This notebook showed how to create two subprocesses with parameters and return values that make it possible to grasp and place an object. You can now experiment with these subprocesses, e.g., try moving a different object or moving to multiple locations.

Consider wrapping recurring behavior into subprocesses. This avoids duplicating patterns across multiple trees, and any improvements immediately apply to all call sites.